In [ ]:
from setup import *
from datetime import datetime as dt
datestring = dt.now().strftime("%Y-%m-%d")


In [13]:
#df_autoarm = pd.read_csv("./data/raw/autoarm_items.csv")
df_autofrei = pd.read_csv("./data/raw/autofrei_items.csv")
df_hitzeschutz = pd.read_csv("./data/raw/2026-08-07_hitzeschutz_items.csv")
df_entities = pd.read_csv("./data/metadata/all_entities.csv")

In [45]:
df_autofrei["class"] = "autofrei"
df_autoarm["class"] = "autoarm"

In [7]:
#df = pd.concat([df_autoarm, df_autofrei])
df = df_hitzeschutz

In [15]:
#meetings sind einzelne Sitzungen
#proposals sind Vorgänge, denen mehrere Sitzungen zugeordnet sein können.
#Achtung: Nicht alle Kommunen ordnen ihre Sitzungen Proposals zu
print("Anzahl gematchter Chunks pro Gruppentyp:")
df_autofrei["groupType"].value_counts()


Anzahl gematchter Chunks pro Gruppentyp:


groupType
meeting     8019
proposal    4698
Name: count, dtype: int64

In [16]:
def get_unique_meetings(df, groupType = None):
    """gruppiert die Chunks, die zum selben Element gehören.
    Falls groupType angegeben ist, dann wird nur auf diesen Type gefiltert."""
    
    result = pd.DataFrame(columns=["groupKey", "date", "chunkCount", "proposalId", "entityId", "entityLevel"])

    if groupType:
        rows = df[df["groupType"] == groupType]
    else:
        rows = df
    unique_groupKeys = rows["groupKey"].drop_duplicates()

    for groupKey in unique_groupKeys:
        chunk_count = len(df[df["groupKey"] == groupKey])
        first_instance = df[df["groupKey"] == groupKey].iloc[0]

        result.loc[len(result)] = {
            "groupKey": groupKey,
            "date": first_instance["date"],
            "chunkCount": chunk_count,
            "proposalId": first_instance["proposalId"],
            "entityId": first_instance["entityId"],
            "entityLevel": first_instance["entityLevel"],
           # "class": first_instance["class"]
        }

    return result

In [18]:
meetings = get_unique_meetings(df, groupType="meeting")
proposals = get_unique_meetings(df, groupType="proposal")

In [19]:
all_items = get_unique_meetings(df)

In [20]:
def add_entityName(data, entities):
    data["entityName"] = data["entityId"].map(entities.drop_duplicates(subset=["id"]).set_index("id")["name"])
    return data


In [21]:
all_items = add_entityName(all_items, df_entities)

In [22]:
all_items["entityName"].value_counts()

entityName
Wilhelmshaven, Stadt            42
Lüneburg, Hansestadt            35
Oldenburg (Oldenburg), Stadt    32
Wolfsburg, Stadt                26
Osnabrück, Stadt                25
                                ..
Bad Zwischenahn                  1
Goldenstedt                      1
Northeim, Stadt                  1
Sande                            1
Papenteich                       1
Name: count, Length: 223, dtype: int64

In [28]:
all_items.to_csv(f"./data/raw/{datestring}_hitzeschutz_grouped_matches.csv")